In [2]:
import torch
import torch.nn as nn

In [5]:
import pandas as pd

In [3]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)  # binary classification
        )
    
    def forward(self, x):
        return self.model(x)

In [6]:
train = pd.read_csv("train_clean.csv")
test = pd.read_csv("test_clean.csv")

In [10]:
X_train = torch.tensor(train.drop(columns=["income"]).values, dtype=torch.float32)
y_train = torch.tensor(train["income"].values, dtype=torch.float32).view(-1, 1)

X_test = torch.tensor(test.drop(columns=["income"]).values, dtype=torch.float32)
y_test = torch.tensor(test["income"].values, dtype=torch.float32).view(-1, 1)

In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [45]:
def init_gaussian(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.01)
        nn.init.zeros_(m.bias)

In [46]:
def init_constant(m):
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 0.1)
        nn.init.constant_(m.bias, 0.0)

In [47]:
def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

In [50]:
model = MLP(input_dim=X_train.shape[1])

In [57]:
model.apply(init_xavier)

MLP(
  (model): Sequential(
    (0): Linear(in_features=53, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [58]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

MLP(
  (model): Sequential(
    (0): Linear(in_features=53, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [59]:
for epoch in range(30):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 Loss: 0.3687
Epoch 2 Loss: 0.3181
Epoch 3 Loss: 0.3141
Epoch 4 Loss: 0.3112
Epoch 5 Loss: 0.3088
Epoch 6 Loss: 0.3075
Epoch 7 Loss: 0.3054
Epoch 8 Loss: 0.3037
Epoch 9 Loss: 0.3024
Epoch 10 Loss: 0.2998
Epoch 11 Loss: 0.2993
Epoch 12 Loss: 0.2975
Epoch 13 Loss: 0.2962
Epoch 14 Loss: 0.2951
Epoch 15 Loss: 0.2937
Epoch 16 Loss: 0.2913
Epoch 17 Loss: 0.2905
Epoch 18 Loss: 0.2898
Epoch 19 Loss: 0.2884
Epoch 20 Loss: 0.2873
Epoch 21 Loss: 0.2853
Epoch 22 Loss: 0.2850
Epoch 23 Loss: 0.2838
Epoch 24 Loss: 0.2822
Epoch 25 Loss: 0.2812
Epoch 26 Loss: 0.2797
Epoch 27 Loss: 0.2786
Epoch 28 Loss: 0.2775
Epoch 29 Loss: 0.2765
Epoch 30 Loss: 0.2748


In [21]:
from sklearn.metrics import classification_report

In [30]:
model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        total_loss += loss.item()

        # probabilities
        probs = torch.sigmoid(logits)
        preds = (probs > 0.4).float()

        all_preds.extend(preds)
        all_labels.extend(y_batch.cpu().numpy())

print("Test Loss:", total_loss / len(test_loader))
print("Test Accuracy:", correct / total)
print(classification_report(all_labels, all_preds))

Test Loss: 0.34729827705182525
Test Accuracy: 0.859310572323932
              precision    recall  f1-score   support

         0.0       0.92      0.87      0.89      4599
         1.0       0.65      0.75      0.69      1464

    accuracy                           0.84      6063
   macro avg       0.78      0.81      0.79      6063
weighted avg       0.85      0.84      0.84      6063



In [34]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train["income"]
)

weights = torch.tensor(weights, dtype=torch.float32).to(device)

In [37]:
weights

tensor([0.6674, 1.9936])

In [38]:
model2 = MLP(input_dim=X_train.shape[1])

In [39]:
criterion = nn.BCEWithLogitsLoss(pos_weight=weights[1])
optimizer = torch.optim.Adam(model2.parameters(), lr=0.001)

In [40]:
for epoch in range(30):
    model2.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        logits = model2(X_batch)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 Loss: 0.5227
Epoch 2 Loss: 0.4623
Epoch 3 Loss: 0.4561
Epoch 4 Loss: 0.4527
Epoch 5 Loss: 0.4482
Epoch 6 Loss: 0.4464
Epoch 7 Loss: 0.4438
Epoch 8 Loss: 0.4418
Epoch 9 Loss: 0.4404
Epoch 10 Loss: 0.4387
Epoch 11 Loss: 0.4366
Epoch 12 Loss: 0.4346
Epoch 13 Loss: 0.4335
Epoch 14 Loss: 0.4338
Epoch 15 Loss: 0.4309
Epoch 16 Loss: 0.4309
Epoch 17 Loss: 0.4299
Epoch 18 Loss: 0.4275
Epoch 19 Loss: 0.4268
Epoch 20 Loss: 0.4268
Epoch 21 Loss: 0.4249
Epoch 22 Loss: 0.4243
Epoch 23 Loss: 0.4225
Epoch 24 Loss: 0.4209
Epoch 25 Loss: 0.4201
Epoch 26 Loss: 0.4196
Epoch 27 Loss: 0.4184
Epoch 28 Loss: 0.4159
Epoch 29 Loss: 0.4145
Epoch 30 Loss: 0.4138


In [41]:
model2.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model2(X_batch)
        loss = criterion(logits, y_batch)

        total_loss += loss.item()

        # probabilities
        probs = torch.sigmoid(logits)
        preds = (probs > 0.4).float()

        all_preds.extend(preds)
        all_labels.extend(y_batch.cpu().numpy())

print("Test Loss:", total_loss / len(test_loader))
print("Test Accuracy:", correct / total)
print(classification_report(all_labels, all_preds))

Test Loss: 0.4718931886710619
Test Accuracy: 0.859310572323932
              precision    recall  f1-score   support

         0.0       0.94      0.81      0.87      4599
         1.0       0.58      0.83      0.68      1464

    accuracy                           0.82      6063
   macro avg       0.76      0.82      0.78      6063
weighted avg       0.85      0.82      0.82      6063



In [42]:
for name, param in model.named_parameters():
    print(name, param.shape)

model.0.weight torch.Size([64, 53])
model.0.bias torch.Size([64])
model.2.weight torch.Size([32, 64])
model.2.bias torch.Size([32])
model.4.weight torch.Size([1, 32])
model.4.bias torch.Size([1])


In [43]:
model.state_dict()

OrderedDict([('model.0.weight',
              tensor([[ 0.0267, -0.1830, -0.1970,  ...,  0.2084, -0.5210, -0.0357],
                      [-0.2165,  0.1240, -0.0309,  ..., -0.3116, -0.0931,  0.2024],
                      [ 0.1426,  0.0225,  0.0621,  ...,  0.0845,  0.1735, -0.0781],
                      ...,
                      [-0.1818, -0.2228,  0.1145,  ..., -0.2847, -0.2697,  0.0584],
                      [ 0.0888, -0.0633,  0.1444,  ...,  0.1196,  0.1523,  0.1785],
                      [ 0.0890,  0.0814,  0.0262,  ...,  0.1676,  0.4607,  0.0141]])),
             ('model.0.bias',
              tensor([ 0.1586,  0.0603, -0.1232,  0.0628, -0.0465, -0.0869,  0.0872, -0.1645,
                       0.0467, -0.0826, -0.1026,  0.0513,  0.1160,  0.1164,  0.0959, -0.0094,
                      -0.0061,  0.0258, -0.1195,  0.1406,  0.1413, -0.0982,  0.1173,  0.0133,
                       0.0644,  0.2031, -0.0307, -0.0536,  0.0625,  0.0138, -0.0830, -0.0788,
                      -0.031

In [44]:
torch.save(model.state_dict(), "best_model.pth")
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>